In [1]:
import numpy as np

# =============================================================================
# ARCH DAM – 2D Curved Frame Analysis
# Elevation: 400 ft from the top of the dam
# Based on Frame2D direct stiffness method (planar beam elements)
# =============================================================================
#
# MODEL DESCRIPTION
# -----------------
# The arch is a horizontal ring at 400 ft depth, idealised as a series of
# straight Euler-Bernoulli beam elements arranged along a circular arc.
#
# Geometry  : Full arch subtends 2π/3 radians (120°), R = 500 ft
# Section   : thickness t = 347 ft  →  unit-width strip (b = 1 ft) analysed
#             I  = 3 481 826.92 ft⁴   (given, full-section moment of inertia)
#             A  = t × b = 347 ft²    (unit strip cross-section)
# Material  : E  = 3 122 018.58 psi  →  converted to ksf for consistent units
# Loading   : Hydrostatic pressure at 400 ft depth
#             p  = γ_water × h = 0.0624 kcf × 400 ft = 24.96 ksf
#             Applied as radially-inward (compressive) distributed loads
#             converted to equivalent nodal forces.
# Supports  : Both abutment nodes are fully fixed (Ux, Uy, Rz restrained).
#
# SIGN CONVENTION  (global X–Y plane)
#   +X → right,  +Y → up (in plan)
#   Arch crown is placed at (R, 0);  abutments are at ±60° from crown.
#
# OUTPUT UNITS
#   Displacements : ft
#   Forces        : kips
#   Moments       : kip-ft
# =============================================================================


class Node:
    def __init__(self, id, x, y):
        self.id = id
        self.x = x          # ft
        self.y = y          # ft
        self.restraints = [0, 0, 0]   # [Ux, Uy, Rz]
        self.loads      = [0.0, 0.0, 0.0]  # [Fx(kip), Fy(kip), Mz(kip-ft)]


class Element:
    def __init__(self, id, n1, n2, E, A, I):
        self.id = id
        self.n1 = n1
        self.n2 = n2
        self.E  = E   # ksf
        self.A  = A   # ft²
        self.I  = I   # ft⁴


class Frame2D:
    def __init__(self):
        self.nodes    = {}
        self.elements = {}

    # ------------------------------------------------------------------ nodes
    def add_node(self, id, x, y):
        self.nodes[id] = Node(id, x, y)

    def delete_node(self, id):
        if id in self.nodes:
            del self.nodes[id]
        self.elements = {
            k: v for k, v in self.elements.items()
            if v.n1 != id and v.n2 != id
        }

    # -------------------------------------------------------------- elements
    def add_element(self, id, n1, n2, E, A, I):
        self.elements[id] = Element(id, n1, n2, E, A, I)

    def delete_element(self, id):
        if id in self.elements:
            del self.elements[id]

    # ---------------------------------------------------------------- assembly
    def assemble(self):
        n_dof = 3 * len(self.nodes)
        K = np.zeros((n_dof, n_dof))
        F = np.zeros(n_dof)

        node_index = {nid: i for i, nid in enumerate(self.nodes)}

        for elem in self.elements.values():
            n1 = self.nodes[elem.n1]
            n2 = self.nodes[elem.n2]

            x1, y1 = n1.x, n1.y
            x2, y2 = n2.x, n2.y

            L = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            c = (x2 - x1) / L
            s = (y2 - y1) / L

            E, A, I = elem.E, elem.A, elem.I

            k_local = np.array([
                [ A*E/L,            0,           0,  -A*E/L,           0,           0],
                [0,      12*E*I/L**3,  6*E*I/L**2,       0, -12*E*I/L**3,  6*E*I/L**2],
                [0,       6*E*I/L**2,    4*E*I/L,        0,  -6*E*I/L**2,    2*E*I/L],
                [-A*E/L,           0,           0,   A*E/L,           0,           0],
                [0,     -12*E*I/L**3, -6*E*I/L**2,       0,  12*E*I/L**3, -6*E*I/L**2],
                [0,       6*E*I/L**2,    2*E*I/L,        0,  -6*E*I/L**2,    4*E*I/L],
            ])

            T = np.array([
                [ c, s, 0, 0, 0, 0],
                [-s, c, 0, 0, 0, 0],
                [ 0, 0, 1, 0, 0, 0],
                [ 0, 0, 0,  c, s, 0],
                [ 0, 0, 0, -s, c, 0],
                [ 0, 0, 0,  0, 0, 1],
            ])

            k_global = T.T @ k_local @ T

            dof_map = []
            for nid in [elem.n1, elem.n2]:
                idx = node_index[nid] * 3
                dof_map.extend([idx, idx + 1, idx + 2])

            for i in range(6):
                for j in range(6):
                    K[dof_map[i], dof_map[j]] += k_global[i, j]

        for nid, node in self.nodes.items():
            idx = node_index[nid] * 3
            F[idx:idx + 3] = node.loads

        return K, F, node_index

    # ------------------------------------------------------------------ solve
    def solve(self):
        K, F, node_index = self.assemble()

        free_dofs = []
        for nid, node in self.nodes.items():
            idx = node_index[nid] * 3
            for i in range(3):
                if node.restraints[i] == 0:
                    free_dofs.append(idx + i)

        Kff = K[np.ix_(free_dofs, free_dofs)]
        Ff  = F[free_dofs]

        Uf = np.linalg.solve(Kff, Ff)

        U = np.zeros(len(F))
        U[free_dofs] = Uf

        return U, node_index

    # --------------------------------------------------------------- results
    def print_results(self):
        U, node_index = self.solve()

        print("\n" + "="*60)
        print("  NODAL DISPLACEMENTS  (ft, radians)")
        print("="*60)
        for nid in self.nodes:
            idx = node_index[nid] * 3
            ux, uy, rz = U[idx:idx + 3]
            print(f"  Node {nid:3d}: "
                  f"Ux={ux:+.6e} ft   "
                  f"Uy={uy:+.6e} ft   "
                  f"Rz={rz:+.6e} rad")

        print("\n" + "="*60)
        print("  MEMBER END FORCES  (kips, kip-ft)")
        print("="*60)

        for elem in self.elements.values():
            n1 = self.nodes[elem.n1]
            n2 = self.nodes[elem.n2]

            x1, y1 = n1.x, n1.y
            x2, y2 = n2.x, n2.y

            L = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            c = (x2 - x1) / L
            s = (y2 - y1) / L

            E, A, I = elem.E, elem.A, elem.I

            k_local = np.array([
                [ A*E/L,            0,           0,  -A*E/L,           0,           0],
                [0,      12*E*I/L**3,  6*E*I/L**2,       0, -12*E*I/L**3,  6*E*I/L**2],
                [0,       6*E*I/L**2,    4*E*I/L,        0,  -6*E*I/L**2,    2*E*I/L],
                [-A*E/L,           0,           0,   A*E/L,           0,           0],
                [0,     -12*E*I/L**3, -6*E*I/L**2,       0,  12*E*I/L**3, -6*E*I/L**2],
                [0,       6*E*I/L**2,    2*E*I/L,        0,  -6*E*I/L**2,    4*E*I/L],
            ])

            T = np.array([
                [ c, s, 0, 0, 0, 0],
                [-s, c, 0, 0, 0, 0],
                [ 0, 0, 1, 0, 0, 0],
                [ 0, 0, 0,  c, s, 0],
                [ 0, 0, 0, -s, c, 0],
                [ 0, 0, 0,  0, 0, 1],
            ])

            dof_map = []
            for nid in [elem.n1, elem.n2]:
                idx = node_index[nid] * 3
                dof_map.extend([idx, idx + 1, idx + 2])

            Ue_global = U[dof_map]
            Ue_local  = T @ Ue_global
            Fe_local  = k_local @ Ue_local

            # Fe_local already in kips and kip-ft (consistent unit system)
            Fx1, Fy1, M1 = Fe_local[0], Fe_local[1], Fe_local[2]
            Fx2, Fy2, M2 = Fe_local[3], Fe_local[4], Fe_local[5]

            print(f"\n  Element {elem.id:3d}  "
                  f"(Node {elem.n1} → Node {elem.n2}):")
            print(f"    Node {elem.n1}: "
                  f"Fx={Fx1:+10.3f} kip   "
                  f"Fy={Fy1:+10.3f} kip   "
                  f"M={M1:+10.3f} kip-ft")
            print(f"    Node {elem.n2}: "
                  f"Fx={Fx2:+10.3f} kip   "
                  f"Fy={Fy2:+10.3f} kip   "
                  f"M={M2:+10.3f} kip-ft")

        print("\n" + "="*60)
        print("  ABUTMENT REACTIONS  (kips, kip-ft)")
        print("="*60)
        K_full, F_full, node_index = self.assemble()
        U_full, _ = self.solve()
        R = K_full @ U_full - F_full
        for nid, node in self.nodes.items():
            if any(node.restraints):
                idx = node_index[nid] * 3
                rx, ry, rm = R[idx], R[idx+1], R[idx+2]
                print(f"  Node {nid:3d}:  "
                      f"Rx={rx:+10.3f} kip   "
                      f"Ry={ry:+10.3f} kip   "
                      f"Rm={rm:+10.3f} kip-ft")


# =============================================================================
#  ARCH DAM SETUP
# =============================================================================

if __name__ == "__main__":

    # ------------------------------------------------------------------
    # Parameters
    # ------------------------------------------------------------------
    depth       = 400          # ft below reservoir surface
    R           = 500          # ft  – arch radius (to centroidal axis)
    half_angle  = np.pi / 3   # 60° each side  →  total arc = 2π/3 = 120°
    t           = 347          # ft  – arch thickness at this elevation
    I_given     = 3_481_826.92 # ft⁴ – moment of inertia (given)
    E_psi       = 3_122_018.58 # psi – modulus of elasticity
    gamma_w     = 0.0624       # kcf – unit weight of water
    n_elem      = 20           # number of elements (increase for accuracy)

    # Unit conversion: psi → ksf (1 ft² = 144 in²)
    E_ksf = E_psi * 144 / 1000   # ksf

    # Cross-sectional properties (unit-width strip, b = 1 ft)
    A_strip = t * 1.0          # ft²  – unit-width strip area
    I_strip = I_given          # ft⁴  – use the given full-section value

    # Hydrostatic pressure at this depth (uniform along the arch ring)
    p = gamma_w * depth        # ksf

    print("="*60)
    print("  ARCH DAM  –  HORIZONTAL RING ANALYSIS")
    print(f"  Elevation : {depth} ft below reservoir surface")
    print(f"  Radius    : {R} ft")
    print(f"  Half-angle: {np.degrees(half_angle):.1f}°  (total arc = 120°)")
    print(f"  Thickness : {t} ft")
    print(f"  I         : {I_given:,.2f} ft⁴")
    print(f"  E         : {E_psi:,.2f} psi  = {E_ksf:,.2f} ksf")
    print(f"  Hydro. p  : {p:.4f} ksf")
    print("="*60)

    # ------------------------------------------------------------------
    # Build the frame model
    # ------------------------------------------------------------------
    frame = Frame2D()

    # Node angles: from -half_angle to +half_angle
    # Crown is at angle 0 → position (R, 0) in plan
    angles = np.linspace(-half_angle, half_angle, n_elem + 1)

    # Add nodes along the arc
    for i, theta in enumerate(angles):
        nid = i + 1
        x = R * np.cos(theta)
        y = R * np.sin(theta)
        frame.add_node(nid, x, y)

    # Abutment boundary conditions (fully fixed at both ends)
    frame.nodes[1].restraints        = [1, 1, 1]   # left  abutment
    frame.nodes[n_elem + 1].restraints = [1, 1, 1]  # right abutment

    # Add elements
    for i in range(n_elem):
        n1 = i + 1
        n2 = i + 2
        frame.add_element(i + 1, n1, n2, E_ksf, A_strip, I_strip)

    # ------------------------------------------------------------------
    # Apply hydrostatic loads as equivalent nodal forces
    # Distributed load p (ksf) acts radially inward on each element.
    # For each element the tributary arc length = element chord length ≈ arc ds.
    # The load is split equally to both end nodes.
    # Radial inward direction at angle θ is (-cos θ, -sin θ).
    # ------------------------------------------------------------------
    dtheta = 2 * half_angle / n_elem   # arc angle per element
    ds     = R * dtheta                # arc length per element

    # Reset nodal loads
    for node in frame.nodes.values():
        node.loads = [0.0, 0.0, 0.0]

    for i, elem in enumerate(frame.elements.values()):
        n1 = frame.nodes[elem.n1]
        n2 = frame.nodes[elem.n2]

        # Mid-angle for radial direction
        theta_mid = (angles[i] + angles[i + 1]) / 2.0
        # Radial inward unit vector at mid-angle
        nx = -np.cos(theta_mid)
        ny = -np.sin(theta_mid)

        # Total force on this element (per unit height, b=1 ft)
        W_elem = p * ds          # kip

        # Distribute equally to the two end nodes
        Fx_node = 0.5 * W_elem * nx
        Fy_node = 0.5 * W_elem * ny

        n1.loads[0] += Fx_node
        n1.loads[1] += Fy_node
        n2.loads[0] += Fx_node
        n2.loads[1] += Fy_node

    # ------------------------------------------------------------------
    # Print summary and solve
    # ------------------------------------------------------------------
    print(f"\n  Number of elements : {n_elem}")
    print(f"  Number of nodes    : {n_elem + 1}")
    print(f"  Tributary arc/elem : {ds:.4f} ft")
    print(f"  Total applied load : {p * 2 * half_angle * R:.2f} kip "
          f"(= p × full arc length)")

    frame.print_results()

    # ------------------------------------------------------------------
    # Quick sanity check – thrust at crown from thin-arch theory
    # H_crown = p * R  (kip/ft of height, per unit strip)
    # ------------------------------------------------------------------
    H_theory = p * R
    print(f"\n{'='*60}")
    print(f"  THIN-ARCH THEORY CHECK (uniform load, fixed arch)")
    print(f"  Crown thrust H = p × R = {p:.4f} × {R} = {H_theory:.2f} kip")
    print(f"  (Numerical result should be close to this for crown element)")
    print(f"{'='*60}\n")

    # ==================================================================
    # CROWN FORCE EXTRACTION
    # Axial Force, Shear, and Moment at the arch crown
    # using the displacement field and the 6×6 element stiffness matrix
    # ==================================================================
    #
    # The crown is at angle θ = 0, which sits between element 10
    # (nodes 10→11) and element 11 (nodes 11→12).
    # Node 11 is the crown node.
    #
    # PROCEDURE
    # ---------
    # 1. Retrieve the global displacement vector U from the solved model.
    # 2. For each crown-adjacent element (10 and 11):
    #    a. Build the 6×6 local stiffness matrix k_local.
    #    b. Build the transformation matrix T (local ↔ global).
    #    c. Extract the 6-DOF global displacement vector for that element.
    #    d. Rotate to local coordinates: u_local = T @ u_global
    #    e. Compute local end forces: f_local = k_local @ u_local
    #    f. Extract the end forces at the crown node (node 11):
    #       - Axial  N  = f_local[0]  (for elem 10, node 11 is the "far" end → index 3)
    #                     f_local[3]  (for elem 11, node 11 is the "near" end → index 0)
    #       - Shear  V  = f_local[1] / f_local[4] (transverse force)
    #       - Moment M  = f_local[2] / f_local[5]
    # 3. Average the two elements' results at node 11 for the best estimate.
    #
    # Sign convention (local element axes):
    #   Local x  →  along element axis (node 1 → node 2)
    #   Local y  →  perpendicular to element, 90° CCW from local x
    #   Axial  N > 0  →  tension  (compression is negative)
    #   Shear  V > 0  →  positive per beam convention
    #   Moment M > 0  →  CCW (sagging positive)
    # ==================================================================

    print("\n" + "="*60)
    print("  CROWN FORCE EXTRACTION")
    print("  Axial Force (N), Shear (V), Moment (M) at Crown Node 11")
    print("="*60)

    # Retrieve the full displacement vector and node index map
    U_full, node_index = frame.solve()

    # Crown node
    crown_nid = n_elem // 2 + 1   # = 11 for n_elem=20

    # The two elements that share the crown node
    # elem 10: nodes 10 → 11  (crown is the FAR  end, local indices 3,4,5)
    # elem 11: nodes 11 → 12  (crown is the NEAR end, local indices 0,1,2)
    crown_elem_ids = [n_elem // 2, n_elem // 2 + 1]   # [10, 11]
    crown_end      = ["far", "near"]

    results_at_crown = []

    for elem_id, end_label in zip(crown_elem_ids, crown_end):
        elem = frame.elements[elem_id]
        n1   = frame.nodes[elem.n1]
        n2   = frame.nodes[elem.n2]

        x1, y1 = n1.x, n1.y
        x2, y2 = n2.x, n2.y

        L = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        c = (x2 - x1) / L
        s = (y2 - y1) / L

        E_e, A_e, I_e = elem.E, elem.A, elem.I

        # 6×6 local stiffness matrix (same formulation as assemble/solve)
        k_local = np.array([
            [ A_e*E_e/L,              0,             0,  -A_e*E_e/L,             0,             0],
            [0,          12*E_e*I_e/L**3,  6*E_e*I_e/L**2,          0, -12*E_e*I_e/L**3,  6*E_e*I_e/L**2],
            [0,           6*E_e*I_e/L**2,    4*E_e*I_e/L,           0,  -6*E_e*I_e/L**2,    2*E_e*I_e/L],
            [-A_e*E_e/L,             0,             0,   A_e*E_e/L,             0,             0],
            [0,         -12*E_e*I_e/L**3, -6*E_e*I_e/L**2,          0,  12*E_e*I_e/L**3, -6*E_e*I_e/L**2],
            [0,           6*E_e*I_e/L**2,    2*E_e*I_e/L,           0,  -6*E_e*I_e/L**2,    4*E_e*I_e/L],
        ])

        # Transformation matrix (global → local)
        T = np.array([
            [ c, s, 0, 0, 0, 0],
            [-s, c, 0, 0, 0, 0],
            [ 0, 0, 1, 0, 0, 0],
            [ 0, 0, 0,  c, s, 0],
            [ 0, 0, 0, -s, c, 0],
            [ 0, 0, 0,  0, 0, 1],
        ])

        # Global DOF indices for this element
        dof_map = []
        for nid in [elem.n1, elem.n2]:
            idx = node_index[nid] * 3
            dof_map.extend([idx, idx + 1, idx + 2])

        # Step 1: Extract global displacements for this element
        u_global = U_full[dof_map]

        # Step 2: Rotate to local coordinates
        u_local = T @ u_global

        # Step 3: Compute local end forces  f = k_local @ u_local
        f_local = k_local @ u_local

        # Step 4: Extract forces at the crown node
        if end_label == "far":
            # Node 11 is the far end of element 10 → local DOF indices 3,4,5
            N = f_local[3]   # axial   (local x, far end)
            V = f_local[4]   # shear   (local y, far end)
            M = f_local[5]   # moment  (rotation, far end)
        else:
            # Node 11 is the near end of element 11 → local DOF indices 0,1,2
            N = f_local[0]   # axial   (local x, near end)
            V = f_local[1]   # shear   (local y, near end)
            M = f_local[2]   # moment  (rotation, near end)

        results_at_crown.append((elem_id, end_label, N, V, M))

        print(f"\n  From Element {elem_id}  ({end_label} end = crown node {crown_nid}):")
        print(f"    Local displacement vector u_local:")
        labels = ["u1(axial,near)", "v1(shear,near)", "θ1(rot,near)",
                  "u2(axial,far )", "v2(shear,far )", "θ2(rot,far )"]
        for lbl, val in zip(labels, u_local):
            print(f"      {lbl}: {val:+.6e} ft/rad")

        print(f"    Local force vector f_local = k_local @ u_local:")
        f_labels = ["N1(axial,near)", "V1(shear,near)", "M1(mom,near)",
                    "N2(axial,far )", "V2(shear,far )", "M2(mom,far )"]
        for lbl, val in zip(f_labels, f_local):
            print(f"      {lbl}: {val:+.4f} kip or kip-ft")

        print(f"\n    >>> At Crown Node {crown_nid}:")
        print(f"        Axial Force  N = {N:+10.3f} kip      "
              f"({'compression' if N < 0 else 'tension'})")
        print(f"        Shear Force  V = {V:+10.3f} kip")
        print(f"        Moment       M = {M:+10.3f} kip-ft")

    # ------------------------------------------------------------------
    # Physical crown forces.
    #
    # The crown node (node 11) sits between element 10 (nodes 10→11)
    # and element 11 (nodes 11→12).  The two local x-axes point in
    # OPPOSITE directions across the crown, so raw averaging cancels.
    #
    # For element 10 (crown is FAR end, local DOF indices 3,4,5):
    #   N2_e10 > 0 means the element pulls toward its near end (tension),
    #   so the crown face is in compression. Thrust = −N2_e10.
    #
    # For element 11 (crown is NEAR end, local DOF indices 0,1,2):
    #   N1_e11 > 0 means the element pushes away from the crown (tension),
    #   so the crown face again sees compression. Thrust = N1_e11.
    #
    # By nodal equilibrium (no external load at crown):
    #   −N2_e10 + N1_e11 = 0  → they are equal. We use elem 11 near-end.
    # ------------------------------------------------------------------

    N_e10_far  = results_at_crown[0][2]
    V_e10_far  = results_at_crown[0][3]
    M_e10_far  = results_at_crown[0][4]

    N_e11_near = results_at_crown[1][2]
    V_e11_near = results_at_crown[1][3]
    M_e11_near = results_at_crown[1][4]

    # Compressive thrust in the arch ring (positive value = compression)
    N_crown = abs(N_e11_near)
    # Shear at crown (by symmetry both elements give the same magnitude)
    V_crown = abs(V_e11_near)
    # Moment at the crown cross-section (near-end of elem 11)
    M_crown = M_e11_near

    print(f"\n{'='*60}")
    print(f"  CROWN SECTION FORCES  (at Node {crown_nid})")
    print(f"{'='*60}")
    print(f"  Arch Ring Thrust (compression)  N = {N_crown:10.3f} kip")
    print(f"  Shear Force                     V = {V_crown:+10.3f} kip")
    print(f"  Bending Moment                  M = {M_crown:+10.3f} kip-ft")
    print(f"\n  Equilibrium check at crown node:")
    print(f"    N_e10_far + N_e11_near = {N_e10_far + N_e11_near:+.6f} kip  (should ≈ 0)")
    print(f"    (elem 10 far-end N is negative = compression, elem 11 near-end N is positive = tension")
    print(f"     in their respective local frames; they balance = same compressive thrust {abs(N_e11_near):.3f} kip)")

    # Stress check at crown section (unit-width strip, b = 1 ft)
    y_extreme = t / 2.0    # ft – extreme fiber distance from centroid
    A_cs      = A_strip    # ft²
    I_cs      = I_strip    # ft⁴

    sigma_axial   = -N_crown / A_cs              # ksf (negative = compression)
    sigma_bending =  abs(M_crown) * y_extreme / I_cs  # ksf (magnitude)
    sigma_max     = sigma_axial + sigma_bending   # downstream face
    sigma_min     = sigma_axial - sigma_bending   # upstream face

    print(f"\n{'='*60}")
    print(f"  CROWN STRESS CHECK  (unit-width strip, b = 1 ft)")
    print(f"{'='*60}")
    print(f"  Section:  A = {A_cs:.1f} ft²   I = {I_cs:,.2f} ft⁴   c = {y_extreme:.1f} ft")
    print(f"  Axial stress   σ_N = -N/A         = {sigma_axial:+.6f} ksf"
          f"  ({sigma_axial * 1000 / 144:+.4f} psi)")
    print(f"  Bending stress σ_M = |M|·c/I      = {sigma_bending:+.6f} ksf"
          f"  ({sigma_bending * 1000 / 144:+.4f} psi)")
    print(f"\n  Combined fiber stresses:")
    print(f"  Downstream face σ = σ_N + σ_M = {sigma_max:+.6f} ksf"
          f"  ({sigma_max * 1000 / 144:+.4f} psi)")
    print(f"  Upstream   face σ = σ_N - σ_M = {sigma_min:+.6f} ksf"
          f"  ({sigma_min * 1000 / 144:+.4f} psi)")
    print(f"{'='*60}\n")

    # ==================================================================
    # PRINCIPAL STRESS ANALYSIS AT THE ARCH CROWN
    # ==================================================================
    #
    # OVERVIEW
    # --------
    # The arch crown cross-section carries three stress resultants:
    #   N  – axial (hoop) thrust, compressive → σ_x  (tangential direction)
    #   V  – transverse shear force            → τ_xy (radial-tangential)
    #   M  – bending moment                    → additional σ_x (linear)
    #
    # We define a 2-D plane-stress state at several points through the
    # thickness (y = −c to +c, measured from the centroid):
    #
    #   σ_x(y)  = axial + bending = −N/A  ±  M·y/I   [ksf, tangential]
    #   σ_y     = 0   (no stress normal to the arch ring in a 2-D ring
    #                  analysis; a full 3-D dam analysis would include
    #                  vertical σ_z from gravity and cantilever action)
    #   τ_xy(y) = V·Q(y) / (I·b)                      [ksf, shear]
    #
    # where Q(y) is the first moment of area of the section above y:
    #   Q(y) = b/2 · (c² − y²)   for a rectangular section of width b
    #
    # Principal stresses at each point:
    #   σ_1,2 = (σ_x + σ_y)/2  ±  √[ ((σ_x − σ_y)/2)² + τ_xy² ]
    #
    # Principal angle (CCW from the tangential axis):
    #   θ_p = 0.5 · arctan( 2·τ_xy / (σ_x − σ_y) )
    #
    # SIGN CONVENTION
    #   σ > 0  →  tension     σ < 0  →  compression
    #   θ_p    →  angle from local x (tangential) to σ_1 plane, CCW +
    # ==================================================================

    print(f"\n{'='*60}")
    print(f"  PRINCIPAL STRESS ANALYSIS AT ARCH CROWN")
    print(f"  (plane-stress state, 2-D ring, unit-width strip b = 1 ft)")
    print(f"{'='*60}")

    b = 1.0        # ft – unit width strip
    A_cs  = A_strip
    I_cs  = I_strip
    c_ext = y_extreme   # = t/2

    # -----------------------------------------------------------------
    # Evaluation points through the thickness
    # y = 0 (centroid), ±c/4, ±c/2, ±3c/4, ±c (extreme fibers)
    # -----------------------------------------------------------------
    fractions = [-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0]
    y_points  = [f * c_ext for f in fractions]

    # Store results for summary table
    rows = []

    print(f"\n  {'y/c':>6}  {'y (ft)':>9}  "
          f"{'σ_x (ksf)':>11}  {'τ_xy (ksf)':>11}  "
          f"{'σ1 (ksf)':>10}  {'σ2 (ksf)':>10}  "
          f"{'θ_p (°)':>8}  {'τ_max (ksf)':>11}")
    print("  " + "-"*86)

    for y in y_points:
        # --- Normal stress (tangential) ---
        # Compression from thrust + bending variation
        # Note: M_crown > 0 means tension on the +y (downstream) face
        sigma_x = (-N_crown / A_cs) + (M_crown * y / I_cs)   # ksf

        # --- Shear stress (parabolic distribution) ---
        # Q(y) = first moment of area of strip from y to +c
        # For rectangle width b: Q = b*(c-y)*(c+y)/2 = b/2*(c²-y²)
        Q_y     = (b / 2.0) * (c_ext**2 - y**2)   # ft³
        tau_xy  = (V_crown * Q_y) / (I_cs * b)     # ksf

        # --- Principal stresses ---
        sigma_y = 0.0   # plane stress (no through-thickness stress in ring model)
        s_avg   = (sigma_x + sigma_y) / 2.0
        r       = np.sqrt(((sigma_x - sigma_y) / 2.0)**2 + tau_xy**2)

        sigma_1 = s_avg + r   # major principal stress
        sigma_2 = s_avg - r   # minor principal stress
        tau_max = r           # maximum shear stress

        # Principal angle: angle from tangential axis (local x) to σ_1 plane
        if abs(sigma_x - sigma_y) < 1e-14:
            theta_p = 45.0 if tau_xy >= 0 else -45.0
        else:
            theta_p = np.degrees(0.5 * np.arctan2(2.0 * tau_xy,
                                                    sigma_x - sigma_y))

        y_over_c = y / c_ext if c_ext != 0 else 0.0
        rows.append((y_over_c, y, sigma_x, tau_xy,
                     sigma_1, sigma_2, theta_p, tau_max))

        print(f"  {y_over_c:>6.2f}  {y:>9.2f}  "
              f"{sigma_x:>+11.4f}  {tau_xy:>+11.6f}  "
              f"{sigma_1:>+10.4f}  {sigma_2:>+10.4f}  "
              f"{theta_p:>+8.2f}  {tau_max:>+11.6f}")

    # -----------------------------------------------------------------
    # Convert extreme fiber results to psi for reporting
    # -----------------------------------------------------------------
    row_top = rows[0]   # y/c = −1.0  (upstream face)
    row_bot = rows[-1]  # y/c = +1.0  (downstream face)
    row_cen = rows[4]   # y/c =  0.0  (centroid)

    def to_psi(ksf):
        return ksf * 1000.0 / 144.0

    print(f"\n{'='*60}")
    print(f"  PRINCIPAL STRESS SUMMARY  (psi)")
    print(f"{'='*60}")

    for label, row in [("Upstream face   (y/c = −1)", row_top),
                        ("Centroid        (y/c =  0)", row_cen),
                        ("Downstream face (y/c = +1)", row_bot)]:
        _, y_val, sx, txy, s1, s2, th, tmax = row
        print(f"\n  {label}:")
        print(f"    σ_x  = {to_psi(sx):+9.2f} psi   "
              f"({'compression' if sx < 0 else 'tension'})")
        print(f"    τ_xy = {to_psi(txy):+9.4f} psi")
        print(f"    σ_1  = {to_psi(s1):+9.2f} psi   "
              f"({'compression' if s1 < 0 else 'tension'})  "
              f"at θ_p = {th:+.2f}° from tangential axis")
        print(f"    σ_2  = {to_psi(s2):+9.2f} psi   "
              f"({'compression' if s2 < 0 else 'tension'})")
        print(f"    τ_max= {to_psi(tmax):+9.4f} psi")

    # -----------------------------------------------------------------
    # Governing values across the entire section
    # -----------------------------------------------------------------
    all_s1   = [r[4] for r in rows]
    all_s2   = [r[5] for r in rows]
    all_tmax = [r[7] for r in rows]

    s1_max   = max(all_s1, key=abs)
    s2_min   = min(all_s2)
    tmax_gov = max(all_tmax)
    th_gov   = rows[all_tmax.index(tmax_gov)][6]

    print(f"\n{'='*60}")
    print(f"  GOVERNING VALUES ACROSS SECTION THICKNESS")
    print(f"{'='*60}")
    print(f"  Max |σ_1|  = {to_psi(s1_max):+9.2f} psi  "
          f"  at y/c = {rows[all_s1.index(s1_max)][0]:+.2f}")
    print(f"  Min  σ_2   = {to_psi(s2_min):+9.2f} psi  "
          f"  at y/c = {rows[all_s2.index(s2_min)][0]:+.2f}  "
          f"({'compression' if s2_min < 0 else 'tension'})")
    print(f"  Max  τ_max = {to_psi(tmax_gov):+9.4f} psi  "
          f"  at y/c = {rows[all_tmax.index(tmax_gov)][0]:+.2f}  "
          f"(at θ = {th_gov + 45.0:+.2f}° from tangential axis)")

    # -----------------------------------------------------------------
    # Mohr's Circle parameters at the three key locations
    # -----------------------------------------------------------------
    print(f"\n{'='*60}")
    print(f"  MOHR'S CIRCLE PARAMETERS (psi)")
    print(f"  (Centre = (σ_1+σ_2)/2,  Radius = τ_max = (σ_1−σ_2)/2)")
    print(f"{'='*60}")
    for label, row in [("Upstream face", row_top),
                        ("Centroid     ", row_cen),
                        ("Downstream   ", row_bot)]:
        _, y_val, sx, txy, s1, s2, th, tmax = row
        centre = to_psi((s1 + s2) / 2.0)
        radius = to_psi(tmax)
        print(f"  {label}:  Centre = {centre:+9.3f} psi   "
              f"Radius = {radius:9.4f} psi   "
              f"θ_p = {th:+.2f}°")

    print(f"{'='*60}\n")

  ARCH DAM  –  HORIZONTAL RING ANALYSIS
  Elevation : 400 ft below reservoir surface
  Radius    : 500 ft
  Half-angle: 60.0°  (total arc = 120°)
  Thickness : 347 ft
  I         : 3,481,826.92 ft⁴
  E         : 3,122,018.58 psi  = 449,570.68 ksf
  Hydro. p  : 24.9600 ksf

  Number of elements : 20
  Number of nodes    : 21
  Tributary arc/elem : 52.3599 ft
  Total applied load : 26138.05 kip (= p × full arc length)

  NODAL DISPLACEMENTS  (ft, radians)
  Node   1: Ux=+0.000000e+00 ft   Uy=+0.000000e+00 ft   Rz=+0.000000e+00 rad
  Node   2: Ux=-2.840746e-03 ft   Uy=-5.913674e-04 ft   Rz=+3.819283e-05 rad
  Node   3: Ux=-6.492617e-03 ft   Uy=-1.394557e-05 ft   Rz=+6.494650e-05 rad
  Node   4: Ux=-1.082811e-02 ft   Uy=+1.120489e-03 ft   Rz=+8.119241e-05 rad
  Node   5: Ux=-1.560341e-02 ft   Uy=+2.319530e-03 ft   Rz=+8.797710e-05 rad
  Node   6: Ux=-2.049794e-02 ft   Uy=+3.225014e-03 ft   Rz=+8.645074e-05 rad
  Node   7: Ux=-2.515480e-02 ft   Uy=+3.617712e-03 ft   Rz=+7.785460e-05 rad
  N